# 02_pretrain — shared external storage

목적: VGGFace2에서 1024D backbone과 ArcFace head의 공통 부모 모델을 학습합니다.

UCFace explicit settings take priority; otherwise FaceX-Zoo conventional training.
Target-only FP32 zero-radicand safeguard; 112×112 inputs; full training lists and final batches.
VGGFace2: provided five-point landmarks + original FaceX-Zoo cropper.
TinyFace/SurvFace/SCface: user-approved resize-only, default bicubic; no face re-detection.
Existing failed-condition checkpoints are preserved as records. 00 saves settings; 01–04 start with EXECUTE=False.


## 1. 설정
공통 변경은 00, 이 단계만 변경하려면 아래 변수를 편집하세요.

선택지·범위가 적힌 변수만 필요한 조건으로 조정합니다. 공통 변경은 00과 같은 SETTINGS_PATH를 사용하는 후속 단계에 일관되게 반영합니다.


In [ ]:
# 목적: 공통 부모의 seed·epoch와 선택적 smoke/속도 점검·재개 checkpoint를 지정합니다.

from pathlib import Path
import sys, json, os
BASE_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fine-tune/modules").is_dir())  # 저장소 위치 자동 탐색값입니다. 직접 변경하지 않습니다.
DATASET_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets" if os.name == "nt" else "/data/face_datasets"))  # 공유 데이터 루트; 환경변수 FACE_DATASET_ROOT 또는 이 Path를 실제 경로로 지정합니다.
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models" if os.name == "nt" else "/data/models"))  # 공유 모델 루트; 환경변수 FACE_MODEL_ROOT 또는 이 Path를 실제 경로로 지정합니다.
SETTINGS_PATH = DATASET_ROOT / ".state/facex/shared_v1_boundary/notebook_settings_mobilefacenet.json"  # 00에서 저장하고 01~04에서 함께 읽을 JSON입니다. backbone/실험별로 파일을 구분합니다.
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
from modules.finetune_protocol import default_profiles
saved = resolve_settings(BASE_DIR, SETTINGS_PATH, require_saved=True)
SEEDS = list(saved["project"]["seed_list"])  # 중복 없는 정수 목록(0 <= seed < 2**32); 일부 seed만 선택 가능하며 후속 단계도 동일하게 맞춥니다.
HEADS = list(saved["campaign"]["heads"])  # "arcface", "adaface", "magface", "cosface" 중 중복 없이 선택; 02 부모는 ArcFace로 고정입니다.
PARENT_SEED = saved["campaign"]["parent_seed"]  # 공통 부모 seed(0 <= seed < 2**32 정수); 후속 head/seed가 공유하므로 02~04에서 유지합니다.
PRETRAIN_EPOCHS = saved["campaign"]["pretrain_epochs"]  # 부모 학습 epoch 수(양의 정수, 기본 40); 재사용 부모의 완료 epoch와 일치해야 합니다.
PRETRAIN_LR = saved["campaign"]["pretrain_lr"]  # parent의 기존 LR을 별도 유지합니다. downstream 논문 LR과 구분합니다.
PARENT_CHECKPOINT = saved["campaign"]["parent_checkpoint"]  # None=02의 고정 경로; 다른 완료 부모를 쓰려면 정확한 경로와 PARENT_SHA256를 함께 지정합니다.
PARENT_SHA256 = saved["campaign"]["parent_sha256"]  # None 또는 지정한 부모 파일의 64자리 SHA-256; 경로와 함께 지정하며 학습 조건도 검증합니다.
FINETUNE_DATASETS = list(saved["campaign"]["finetune_datasets"])  # "survface", "tinyface", "scface" 중 선택합니다. LFW/RFW는 평가 전용입니다.
FINETUNE_PROFILES = saved["finetune_profiles"]  # 외부 protocol의 분할·경로·hash입니다. 분할 인물/수량/hash를 여기서 임의로 고치지 않습니다.
# 원본 데이터·공유 전처리·부모·과거 실패 기록은 이 보존 옵션의 삭제 대상이 아닙니다.
KEEP_RAW_RESULTS = saved["project"]["keep_raw_results"]  # True=이번 학습의 주기별 checkpoint 보존, False=학습 중 오래된 중간 checkpoint 정리.
NUM_WORKERS = saved["training"]["num_workers"]  # DataLoader worker 수(0~12 정수); 0=현재 프로세스에서 로딩, 큰 값은 CPU/RAM을 더 사용합니다.
PREFETCH_FACTOR = saved["training"]["prefetch_factor"]  # worker당 미리 읽을 batch 수(1 이상 정수); NUM_WORKERS>0일 때 적용합니다.
PERSISTENT_WORKERS = saved["training"]["persistent_workers"]  # True=epoch 사이 worker 유지, False=다시 생성; NUM_WORKERS=0이면 적용되지 않습니다.
TORCH_THREADS = saved["training"]["torch_threads"]  # PyTorch CPU 연산 thread 수(1 이상 정수); GPU 수나 DataLoader worker 수와 다릅니다.
LOSS_SYNC_BATCHES = saved["training"].get("loss_sync_batches", 20)  # loss 집계값을 CPU로 가져오는 batch 간격(1 이상 정수); 매 batch 학습 업데이트는 유지합니다.
RECOVERY_MINUTES = saved["training"]["recovery_minutes"]  # 재개용 checkpoint 저장 간격(분, 0보다 큰 값); 짧을수록 복구 간격과 디스크 쓰기가 촘촘해집니다.
DEVICE = saved["project"]["device"]  # "cuda:0" 또는 사용 가능한 GPU 번호("cuda:1" 등); 정식 학습은 CUDA 필수이며 CPU fallback은 없습니다.
DATASET = dict(saved["dataset"])  # VGGFace2 경로·입력 설정입니다. 112×112와 pixel_mean=127.5/pixel_std=128.0 조건을 유지합니다.
BACKBONE = dict(saved["backbone"])  # 00에서 MobileFaceNet/Resnet IR-50 선택; 1024D·dropout 0.6·out_h/out_w=7 조건을 유지합니다.
# 정식 fine-tune: optimizer=adam, lr=MobileFaceNet 0.001 / ResNet-50·100 0.0001, epoches=40, milestones=[10,20,30], lr_decay_ratio=0.1 고정.
# baseline batch_size=64는 MobileFaceNet·ResNet-50/100 공통; save_freq는 1 이상 epoch, drop_last=False 유지.
TRAINING = dict(saved["training"])  # 학습/저장 설정 사전입니다. 위 고정 학습 조건과 조절 가능한 저장 간격을 구분합니다.
HEAD_PARAMETERS = saved["heads"]  # head별 margin/scale 기준값입니다. 정식 fine-tune은 이를 검증하므로 head 선택은 HEADS에서 합니다.
PARENT_HEAD = dict(saved["parent_head"])  # 부모는 ArcFace, margin_arc=0.35/margin_am=0/scale=32로 고정합니다.
# flip_probability/affine_probability: 0~1, rotation_degrees: 0 이상, scale_range=[low,high]: 0<low<=high.
# border_mode="reflect"/"constant", center="image_center" 고정; 현재 수치를 UCFace 명시값으로 간주하지 않습니다.
AUGMENTATION = dict(saved["augmentation"])  # 증강 수치 변경은 별도 학습 조건입니다. 새 namespace와 부모 학습으로 구분합니다.
ALIGNMENT = dict(saved["alignment"])  # VGGFace2=official_landmarks, 저해상도=resize_only; 보간법은 bicubic/bilinear 중 선택합니다.
PATHS = dict(saved["paths"])  # checkpoint/export/log/report/cache 경로; 00에서 namespace별로 정하고 후속 단계가 이어받습니다.
REUSE_COMPLETED = saved["campaign"]["reuse_completed"]  # True=동일 조건의 검증된 완료 결과 재사용; False=완료 경로가 있으면 중단하며 덮어쓰지 않습니다.
EXECUTE = False  # False=정식 학습 안 함, True=부모 학습/smoke 실행; RUN_BENCHMARK는 별도 실행 스위치입니다.
STAGE = "pretrain"  # 이 노트북의 고정 단계입니다. 다른 단계는 해당 번호의 노트북을 사용합니다.
RUN_SMOKE_TEST = False  # False=정식 학습, True=합성 이미지 1 step 점검; EXECUTE=True 필요, 정식 완료 모델은 생성하지 않습니다.
RESUME_CHECKPOINTS = {}  # {}=자동 재개 안 함; {"run_id": "checkpoint 경로"}로 현재 namespace의 동일 조건 중단 run을 지정합니다.
RUN_BENCHMARK = False  # True=EXECUTE와 무관하게 GPU 속도 점검 실행, 정식 학습 건너뜀; False=속도 점검 안 함.
WORKER_CANDIDATES = [0,2,4]  # 속도 점검에서 비교할 worker 수 목록(예: [0, 2, 4], 각 0~12 정수)입니다.
BENCHMARK_STEPS = 100  # worker 후보당 측정 batch 수(양의 정수); loader가 추가 warm-up 10 batch까지 공급해야 합니다.
# 위 변수들을 실제 모듈 설정으로 묶어 검증합니다. 아래 전달부는 직접 편집할 필요가 없습니다.
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
 "project":dict(seed=SEEDS[0],seed_list=SEEDS,keep_raw_results=KEEP_RAW_RESULTS,device=DEVICE),
 "campaign":dict(heads=HEADS,reuse_completed=REUSE_COMPLETED,parent_seed=PARENT_SEED,
    pretrain_epochs=PRETRAIN_EPOCHS,pretrain_lr=PRETRAIN_LR,parent_checkpoint=PARENT_CHECKPOINT,parent_sha256=PARENT_SHA256,
    finetune_datasets=FINETUNE_DATASETS),
 "finetune_profiles":FINETUNE_PROFILES,"dataset":DATASET,"backbone":BACKBONE,"heads":HEAD_PARAMETERS,
 "training":{**TRAINING,"num_workers":NUM_WORKERS,"prefetch_factor":PREFETCH_FACTOR,
    "persistent_workers":PERSISTENT_WORKERS,"torch_threads":TORCH_THREADS,"recovery_minutes":RECOVERY_MINUTES,"loss_sync_batches":LOSS_SYNC_BATCHES},
 "paths":PATHS,"parent_head":PARENT_HEAD,"augmentation":AUGMENTATION,"alignment":ALIGNMENT})


## 2. 계획

In [ ]:
# 목적: 생성할 공통 ArcFace 부모의 run ID와 실제 실행 여부를 확인합니다.

plan = plan_matrix(settings, STAGE)
for row in plan:
    print(row)
print("예정:",len(plan),"EXECUTE:",EXECUTE,"SMOKE:",RUN_SMOKE_TEST)

## 3. 공통 부모와 선택적 속도 점검

RUN_BENCHMARK=True는 EXECUTE=False여도 실제 GPU 연산을 수행합니다. 전체 전처리가 완료된 입력이 필요하며 정식 모델을 생성하지 않습니다.


In [ ]:
# 목적: 공통 부모 설정을 확인하고 RUN_BENCHMARK=True일 때 GPU의 worker별 처리 속도를 측정합니다.

print("공통 부모:", PARENT_CHECKPOINT or "02의 PARENT_SEED 고정 경로", "parent seed:", PARENT_SEED)
print("시작 전 00 설정 저장과 01 데이터 준비를 확인하세요.")
from modules.process_runner import launch_campaign
if RUN_BENCHMARK:
    benchmark_results = launch_campaign(BASE_DIR,settings,"benchmark",benchmark_workers=WORKER_CANDIDATES,benchmark_steps=BENCHMARK_STEPS)
    print(benchmark_results)

## 4. 별도 프로세스 실행
노트북 Interrupt는 stop 파일을 만들고 학습은 optimizer step 완료 후 재개 checkpoint를 저장합니다. Kernel 강제 종료는 이 절차를 실행하지 않을 수 있으므로 출력된 stop 파일 경로 또는 progress.json을 확인하세요. 이 부모 학습의 epoch 수는 PRETRAIN_EPOCHS를, batch 크기·optimizer·학습률은 TRAINING을 따릅니다.

In [ ]:
# 목적: EXECUTE=True이고 benchmark 모드가 아니면 별도 프로세스로 부모 학습 또는 smoke 점검을 실행합니다.

from modules.process_runner import launch_campaign
results = []
if EXECUTE and not RUN_BENCHMARK:
    results=launch_campaign(BASE_DIR,settings,STAGE,RESUME_CHECKPOINTS,smoke=RUN_SMOKE_TEST)
else:
    print("계획 또는 benchmark 모드: 정식 작업 미실행")

## 5. 완료 상태

In [ ]:
# 목적: 부모 학습의 완료·부분·smoke 상태와 실패 시 중단 범위를 확인합니다.

for result in results:
    print(result)
print("정식 완료:",sum(r.get("completed") is True for r in results),"/",len(plan))
print("실패 시 summary에 미실행 범위를 기록하고 후속 작업을 중단합니다.")
print("해석 안내:",BASE_DIR/"fine-tune/README.md")